# Continual Pre-Training (CPT) for SmolLM2-135M
## Improving MMLU Score through Knowledge Enhancement

This notebook implements a complete CPT pipeline to improve SmolLM2-135M's MMLU score.

**Current baseline:** 30.2 (HuggingFace leaderboard)

**Strategy:**
1. Curate knowledge-dense training data (Wikipedia, math, textbooks)
2. Mix with original training distribution to prevent forgetting
3. Train with lower learning rate than initial pretraining
4. Evaluate on MMLU cloze benchmark

## 1. Environment Setup

In [ ]:
# Install required packages
!pip install -q transformers datasets accelerate tokenizers
!pip install -q torch torchvision torchaudio
!pip install -q tensorboard tqdm

# Check GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Configuration
import os

CONFIG = {
    "model_name": "HuggingFaceTB/SmolLM2-135M",
    "data_dir": "./processed_data",
    "output_dir": "./cpt_checkpoints",
    "max_seq_length": 2048,
    "target_samples": 10000,  # Reduced for Colab time limits
    "learning_rate": 5e-5,
    "num_epochs": 1,
    "seed": 42,
}

os.makedirs(CONFIG["data_dir"], exist_ok=True)
os.makedirs(CONFIG["output_dir"], exist_ok=True)

## 2. Baseline Evaluation

First, let's verify the baseline MMLU score.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm

def evaluate_mmlu_quick(model_path, n_subjects=10, samples_per_subject=50):
    """
    Quick MMLU evaluation for fast iteration.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    print(f"Loading model: {model_path}")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()
    
    # Select diverse subjects
    subjects = [
        "elementary_mathematics",
        "high_school_physics",
        "college_biology",
        "world_religions",
        "high_school_us_history",
        "professional_psychology",
        "business_ethics",
        "computer_security",
        "anatomy",
        "philosophy",
    ][:n_subjects]
    
    total_correct = 0
    total_questions = 0
    results = {}
    
    for subject in tqdm(subjects, desc="Evaluating"):
        try:
            dataset = load_dataset("cais/mmlu", subject, split="test", trust_remote_code=True)
            dataset = dataset.select(range(min(samples_per_subject, len(dataset))))
        except:
            continue
        
        correct = 0
        for example in dataset:
            # Format prompt
            prompt = f"Question: {example['question']}\n"
            for i, choice in enumerate(example['choices']):
                prompt += f"{['A', 'B', 'C', 'D'][i]}. {choice}\n"
            prompt += "Answer:"
            
            # Get model prediction
            inputs = tokenizer(prompt, return_tensors="pt").to(device)
            with torch.no_grad():
                outputs = model(**inputs)
                logits = outputs.logits[0, -1, :]
            
            # Get token probabilities for A, B, C, D
            choice_tokens = [tokenizer.encode(f" {c}", add_special_tokens=False)[0] for c in "ABCD"]
            choice_logits = logits[choice_tokens]
            predicted = choice_logits.argmax().item()
            
            if predicted == example['answer']:
                correct += 1
        
        accuracy = correct / len(dataset)
        results[subject] = accuracy
        total_correct += correct
        total_questions += len(dataset)
    
    overall = total_correct / total_questions if total_questions > 0 else 0
    
    print(f"\n{'='*50}")
    print(f"MMLU Score: {overall * 100:.2f}")
    print(f"Total: {total_correct}/{total_questions}")
    print(f"{'='*50}")
    
    for subject, acc in sorted(results.items(), key=lambda x: -x[1]):
        print(f"  {subject}: {acc*100:.1f}%")
    
    return overall * 100, results

# Evaluate baseline
print("Evaluating baseline model...")
baseline_score, baseline_results = evaluate_mmlu_quick(CONFIG["model_name"])

## 3. Data Collection & Processing

In [ ]:
import random
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer

random.seed(CONFIG["seed"])

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer vocab size: {tokenizer.vocab_size}")

In [ ]:
def create_training_data(tokenizer, num_samples=10000, max_length=2048):
    """
    Create training dataset optimized for MMLU improvement.
    Combines Wikipedia (general knowledge) and Synthetic QA pairs.
    """
    samples = []
    
    # --- Wikipedia for general knowledge ---
    print("Loading Wikipedia...")
    wiki_samples = num_samples // 2
    
    try:
        wiki = load_dataset(
            "wikimedia/wikipedia",
            "20231101.en",
            split="train",
            streaming=True,
            trust_remote_code=True,
        )
        
        buffer = []
        collected = 0
        
        for article in tqdm(wiki, total=wiki_samples, desc="Processing Wikipedia"):
            if collected >= wiki_samples:
                break
                
            text = article.get("text", "")
            if len(text) < 200:
                continue
            
            tokens = tokenizer.encode(text, add_special_tokens=False)
            buffer.extend(tokens)
            buffer.append(tokenizer.eos_token_id)
            
            while len(buffer) >= max_length:
                samples.append({
                    "input_ids": buffer[:max_length],
                    "attention_mask": [1] * max_length,
                })
                buffer = buffer[max_length:]
                collected += 1
                
                if collected >= wiki_samples:
                    break
        
        print(f"Collected {collected} Wikipedia samples")
        
    except Exception as e:
        print(f"Wikipedia loading failed: {e}")
        print("Continuing with synthetic data only...")
    
    # --- Synthetic QA for MMLU format ---
    print("\nGenerating synthetic QA samples...")
    qa_samples = num_samples - len(samples)
    
    qa_templates = [
        # STEM
        ("What is the chemical symbol for gold?", "Au (from Latin 'aurum')"),
        ("What is the formula for the area of a circle?", "A = πr², where r is the radius"),
        ("What organelle is responsible for cellular respiration?", "The mitochondria"),
        ("What is Newton's first law of motion?", "An object at rest stays at rest unless acted upon by external force"),
        ("What is the Pythagorean theorem?", "In a right triangle, a² + b² = c²"),
        ("What is the speed of light?", "Approximately 299,792,458 meters per second in vacuum"),
        ("What is DNA?", "Deoxyribonucleic acid, carries genetic instructions"),
        ("What is photosynthesis?", "Process by which plants convert sunlight into chemical energy"),
        # Humanities
        ("Who wrote 'Hamlet'?", "William Shakespeare"),
        ("What year did World War I begin?", "1914"),
        ("Who painted the Sistine Chapel ceiling?", "Michelangelo"),
        ("What is the capital of Japan?", "Tokyo"),
        ("Who was the first Roman Emperor?", "Augustus (Octavian)"),
        # Social Sciences
        ("What is supply and demand?", "Economic model of price determination"),
        ("What is cognitive dissonance?", "Mental discomfort from contradictory beliefs"),
        ("What is GDP?", "Gross Domestic Product, total value of goods and services"),
        ("What is inflation?", "General increase in prices, decrease in purchasing power"),
    ]
    
    formats = [
        "Question: {q}\nAnswer: {a}",
        "Q: {q}\nA: {a}",
        "{q}\n\nThe answer is: {a}",
    ]
    
    for _ in range(qa_samples):
        q, a = random.choice(qa_templates)
        fmt = random.choice(formats)
        text = fmt.format(q=q, a=a)
        
        tokens = tokenizer.encode(text, add_special_tokens=True)
        
        if len(tokens) < max_length:
            padding = max_length - len(tokens)
            samples.append({
                "input_ids": tokens + [tokenizer.pad_token_id] * padding,
                "attention_mask": [1] * len(tokens) + [0] * padding,
            })
    
    print(f"Total samples: {len(samples)}")
    random.shuffle(samples)
    return samples

# Create dataset
samples = create_training_data(tokenizer, num_samples=CONFIG["target_samples"])

In [ ]:
# Convert to HuggingFace dataset and save
from datasets import Dataset

dataset = Dataset.from_dict({
    "input_ids": [s["input_ids"] for s in samples],
    "attention_mask": [s["attention_mask"] for s in samples],
})

# Split
dataset = dataset.train_test_split(test_size=0.02, seed=CONFIG["seed"])

print(f"Train samples: {len(dataset['train'])}")
print(f"Val samples: {len(dataset['test'])}")

# Save
dataset.save_to_disk(CONFIG["data_dir"])
print(f"\nDataset saved to {CONFIG['data_dir']}")

## 4. Continual Pre-Training

In [ ]:
import math
from torch.utils.data import DataLoader
from datasets import load_from_disk
from transformers import get_cosine_schedule_with_warmup

# Load dataset
dataset = load_from_disk(CONFIG["data_dir"])

# Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

# Enable gradient checkpointing for memory efficiency
model.gradient_checkpointing_enable()
model = model.to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training configuration
TRAIN_CONFIG = {
    "batch_size": 2,
    "gradient_accumulation_steps": 8,
    "learning_rate": 5e-5,
    "warmup_ratio": 0.05,
    "max_grad_norm": 1.0,
    "weight_decay": 0.1,
    "num_epochs": 1,
    "eval_steps": 200,
    "save_steps": 500,
    "logging_steps": 50,
}

# DataLoader
def collate_fn(batch):
    input_ids = torch.tensor([x["input_ids"] for x in batch])
    attention_mask = torch.tensor([x["attention_mask"] for x in batch])
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": input_ids.clone(),
    }

train_loader = DataLoader(
    dataset["train"],
    batch_size=TRAIN_CONFIG["batch_size"],
    shuffle=True,
    collate_fn=collate_fn,
)

eval_loader = DataLoader(
    dataset["test"],
    batch_size=TRAIN_CONFIG["batch_size"] * 2,
    shuffle=False,
    collate_fn=collate_fn,
)

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=TRAIN_CONFIG["learning_rate"],
    weight_decay=TRAIN_CONFIG["weight_decay"],
    betas=(0.9, 0.95),
)

# Scheduler
total_steps = (len(train_loader) // TRAIN_CONFIG["gradient_accumulation_steps"]) * TRAIN_CONFIG["num_epochs"]
warmup_steps = int(total_steps * TRAIN_CONFIG["warmup_ratio"])

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f"Total training steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")
print(f"Effective batch size: {TRAIN_CONFIG['batch_size'] * TRAIN_CONFIG['gradient_accumulation_steps']}")

In [ ]:
# Training loop
from torch.cuda.amp import autocast, GradScaler

scaler = GradScaler()
model.train()

global_step = 0
accumulated_loss = 0

progress_bar = tqdm(total=total_steps, desc="Training")

for epoch in range(TRAIN_CONFIG["num_epochs"]):
    print(f"\nEpoch {epoch + 1}/{TRAIN_CONFIG['num_epochs']}")
    
    for batch_idx, batch in enumerate(train_loader):
        # Move to device
        batch = {k: v.to(device) for k, v in batch.items()}
        
        # Forward pass with mixed precision
        with autocast():
            outputs = model(**batch)
            loss = outputs.loss / TRAIN_CONFIG["gradient_accumulation_steps"]
        
        # Backward pass
        scaler.scale(loss).backward()
        accumulated_loss += loss.item()
        
        # Gradient accumulation step
        if (batch_idx + 1) % TRAIN_CONFIG["gradient_accumulation_steps"] == 0:
            # Gradient clipping
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), TRAIN_CONFIG["max_grad_norm"])
            
            # Optimizer step
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            
            global_step += 1
            progress_bar.update(1)
            
            # Logging
            if global_step % TRAIN_CONFIG["logging_steps"] == 0:
                avg_loss = accumulated_loss / TRAIN_CONFIG["logging_steps"]
                lr = scheduler.get_last_lr()[0]
                print(f"Step {global_step} | Loss: {avg_loss:.4f} | LR: {lr:.2e}")
                accumulated_loss = 0
            
            # Save checkpoint
            if global_step % TRAIN_CONFIG["save_steps"] == 0:
                save_path = os.path.join(CONFIG["output_dir"], f"checkpoint-{global_step}")
                model.save_pretrained(save_path)
                tokenizer.save_pretrained(save_path)
                print(f"Saved checkpoint to {save_path}")

progress_bar.close()

# Save final model
final_path = os.path.join(CONFIG["output_dir"], "final")
model.save_pretrained(final_path)
tokenizer.save_pretrained(final_path)
print(f"\nFinal model saved to {final_path}")

## 5. Post-Training Evaluation

In [ ]:
# Evaluate fine-tuned model
print("Evaluating fine-tuned model...")
final_score, final_results = evaluate_mmlu_quick(
    os.path.join(CONFIG["output_dir"], "final")
)

In [ ]:
# Compare results
print("\n" + "="*60)
print("COMPARISON: Baseline vs Fine-tuned")
print("="*60)
print(f"\nBaseline MMLU Score: {baseline_score:.2f}")
print(f"Fine-tuned MMLU Score: {final_score:.2f}")
print(f"Improvement: {final_score - baseline_score:+.2f}")

print("\nPer-subject comparison:")
for subject in baseline_results:
    if subject in final_results:
        diff = (final_results[subject] - baseline_results[subject]) * 100
        print(f"  {subject}: {baseline_results[subject]*100:.1f}% -> {final_results[subject]*100:.1f}% ({diff:+.1f})")

## 6. Full MMLU Evaluation (Optional)

Run this for complete evaluation on all 57 subjects.

In [ ]:
# Full evaluation (takes longer)
# Uncomment to run

# print("Running full MMLU evaluation...")
# full_score, full_results = evaluate_mmlu_quick(
#     os.path.join(CONFIG["output_dir"], "final"),
#     n_subjects=57,  # All subjects
#     samples_per_subject=None,  # All samples
# )

## Summary

This notebook demonstrated:
1. **Baseline evaluation** of SmolLM2-135M on MMLU
2. **Data curation** combining Wikipedia and synthetic QA
3. **CPT training** with memory-efficient techniques
4. **Post-training evaluation** to measure improvement

Key techniques used:
- Lower learning rate (5e-5) vs pretraining
- Gradient checkpointing for memory efficiency
- Mixed precision training (FP16)
- Cosine learning rate schedule with warmup
- Knowledge-dense data curation